# Lab 7.5 &mdash; Challenge: The Release Gate

**Level:** Advanced &middot; challenge &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 3 &middot; Module 7 &mdash; Multi-Agent System Evaluation**

### What you'll do
- Build a gate that fails a build rather than a dashboard nobody reads
- Block the version that is provably better and too expensive
- Find out that a conservative test lets a bad version through, and fix it
- Produce a ship / do-not-ship decision with reasons for four real candidates

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **The deliverable of Day 3 so far.** Four candidates, three thresholds agreed in
> advance, and one decision per candidate that a release manager could act on.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-7-05")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- four candidate versions, already measured
# Each was run 30 times over a 50-case eval set -- the shape Lab 7.1 showed you need before a
# difference is even expressible. Cost is per case; latency is p95 seconds.

CANDIDATES = {
    "v6 (current)": {
        "rates": [0.76, 0.76, 0.66, 0.74, 0.66, 0.76, 0.74, 0.86, 0.74, 0.78, 0.74, 0.74, 0.72,
                  0.8, 0.78, 0.7, 0.76, 0.66, 0.88, 0.88, 0.8, 0.74, 0.62, 0.84, 0.66, 0.8, 0.7,
                  0.64, 0.74, 0.82],
        "cost_per_case": 0.0121, "p95_latency_s": 11.4},
    "v7 better prompt": {
        "rates": [0.82, 0.74, 0.84, 0.8, 0.82, 0.64, 0.84, 0.88, 0.86, 0.86, 0.82, 0.8, 0.8, 0.86,
                  0.88, 0.74, 0.8, 0.68, 0.78, 0.94, 0.74, 0.74, 0.78, 0.82, 0.86, 0.86, 0.82,
                  0.86, 0.86, 0.82],
        "cost_per_case": 0.0129, "p95_latency_s": 11.9},
    "v8 more agents": {
        "rates": [0.98, 0.94, 0.98, 0.94, 0.96, 0.92, 0.94, 0.96, 0.94, 0.98, 0.92, 0.98, 0.96,
                  0.96, 0.94, 0.96, 0.98, 0.96, 0.9, 0.96, 1.0, 0.96, 0.96, 0.94, 0.96, 0.98,
                  0.92, 0.94, 0.96, 0.98],
        "cost_per_case": 0.0402, "p95_latency_s": 19.6},
    "v9 cheaper model": {
        "rates": [0.6, 0.68, 0.54, 0.6, 0.56, 0.64, 0.72, 0.62, 0.66, 0.58, 0.48, 0.54, 0.58,
                  0.52, 0.5, 0.52, 0.6, 0.54, 0.64, 0.44, 0.7, 0.42, 0.66, 0.54, 0.64, 0.46,
                  0.56, 0.58, 0.56, 0.62],
        "cost_per_case": 0.0058, "p95_latency_s": 8.2},
}

CURRENT = "v6 (current)"

# Agreed in a calm week, before anybody had a release they wanted to push.
COST_CEILING    = 0.020     # per case
LATENCY_CEILING = 15.0      # p95 seconds
QUALITY_TOLERANCE = 0.02    # how much mean quality may drop and still be called "no regression"

import statistics
print(f"{len(CANDIDATES)} candidates, 30 runs each; ceilings: "
      f"cost {COST_CEILING}, p95 {LATENCY_CEILING}s")

## Concept

A dashboard reports. A gate decides. The difference is whether the build fails.

Three thresholds, agreed while nobody was under pressure:

- **quality** must not regress against the current version on the same eval set
- **cost per case** must stay under an agreed ceiling
- **p95 latency** must stay under an agreed ceiling

The interesting candidates are the ones that pass two and fail one.

## Section 1 &mdash; Summarise each candidate

A mean and a range, because Lab 7.1 established that a bare number is not reportable.

In [ ]:
def summarise(name: str) -> dict:
    rates = CANDIDATES[name]["rates"]
    # TODO: the mean is what the gate compares; the range is what stops the argument.
    return {"name": name, "mean": round(BLANK, 4),
            "low": min(rates), "high": max(rates),
            "cost": CANDIDATES[name]["cost_per_case"],
            "p95": CANDIDATES[name]["p95_latency_s"]}


def ranges_overlap(a: str, b: str) -> bool:
    ra, rb = CANDIDATES[a]["rates"], CANDIDATES[b]["rates"]
    return not (min(rb) > max(ra) or min(ra) > max(rb))

In [ ]:
# --- Self-check: Section 1
check("the current version averages about 75%",
      lambda: 0.74 < summarise(CURRENT)["mean"] < 0.76)
check("v8 is the strongest on quality",
      lambda: max(CANDIDATES, key=lambda n: summarise(n)["mean"]) == "v8 more agents")
check("v9 is the weakest",
      lambda: min(CANDIDATES, key=lambda n: summarise(n)["mean"]) == "v9 cheaper model")
check("only v8 is PROVABLY better than the current version",
      lambda: [n for n in CANDIDATES if n != CURRENT and not ranges_overlap(CURRENT, n)]
              == ["v8 more agents"],
      "v7 is better on average and its range still overlaps v6's -- not proven, on 30 runs")
check("v9's range overlaps the current version's too",
      lambda: ranges_overlap(CURRENT, "v9 cheaper model") is True,
      "so a test that only blocks PROVEN regressions would let v9 straight through")

def _summary():
    print(f"  {'candidate':20}{'mean':>8}{'range':>14}{'cost':>9}{'p95':>7}")
    print("  " + "-" * 60)
    for n in CANDIDATES:
        s = summarise(n)
        print(f"  {n:20}{s['mean']:>8.1%}{s['low']:>7.0%}-{s['high']:<6.0%}"
              f"{s['cost']:>9.4f}{s['p95']:>7.1f}")
guard(_summary)

## Section 2 &mdash; The gate

Three checks and a list of reasons. A gate that says only &ldquo;blocked&rdquo; gets overridden; one that
says *why* gets fixed.

In [ ]:
def gate(name: str, current: str = CURRENT) -> dict:
    """Should this version ship? Returns a decision and every reason against it."""
    s, cur = summarise(name), summarise(current)
    reasons = []
    # TODO: three checks. Quality must not drop by more than QUALITY_TOLERANCE against the
    # current version's mean; cost and p95 must stay under their ceilings. Append one
    # human-readable reason per failure.
    BLANK
    return {"name": name, "ship": not reasons, "reasons": reasons}

In [ ]:
# --- Self-check: Section 2
check("the current version passes its own gate",
      lambda: gate(CURRENT)["ship"] is True,
      "a gate the incumbent fails is a gate nobody will agree to")
check("v7 ships: no regression, and both ceilings respected",
      lambda: gate("v7 better prompt")["ship"] is True)
check("V8 IS BLOCKED, and it is the best version on quality",
      lambda: gate("v8 more agents")["ship"] is False,
      "provably better, 3.3x the cost and over the latency ceiling -- the gate does its job here")
check("and it is blocked for two separate reasons",
      lambda: len(gate("v8 more agents")["reasons"]) == 2)
check("v9 is blocked on quality",
      lambda: gate("v9 cheaper model")["ship"] is False
              and "quality" in gate("v9 cheaper model")["reasons"][0])
check("v9 would have passed a gate that only blocked PROVEN regressions",
      lambda: ranges_overlap(CURRENT, "v9 cheaper model") is True,
      "the conservative test refuses to confirm anything, including that this is worse")
check("every blocked candidate says why",
      lambda: all(gate(n)["reasons"] for n in CANDIDATES if not gate(n)["ship"]))

def _decisions():
    for n in CANDIDATES:
        g = gate(n)
        print(f"  {'SHIP  ' if g['ship'] else 'BLOCK '} {n}")
        for r in g["reasons"]:
            print(f"           - {r}")
guard(_decisions)

## Section 3 &mdash; The two ways this gate can be wrong

Both matter, and they are not symmetric.

In [ ]:
def false_pass_risk() -> dict:
    """A version that is worse and ships anyway."""
    worse = "v9 cheaper model"
    strict_only = ranges_overlap(CURRENT, worse)   # a "proven regression" test would not block it
    return {"version": worse,
            "blocked_by_mean_test": not gate(worse)["ship"],
            "would_pass_proven_regression_test": strict_only}


def false_block_risk() -> dict:
    """A version that is better and does not ship."""
    better = "v8 more agents"
    return {"version": better,
            "provably_better": not ranges_overlap(CURRENT, better),
            "blocked": not gate(better)["ship"],
            "reasons": gate(better)["reasons"]}


def cost_of_shipping(name: str, cases_per_day: int = 20000) -> float:
    """What choosing this version costs per day, which is what the argument is really about."""
    return round(CANDIDATES[name]["cost_per_case"] * cases_per_day, 2)

In [ ]:
# --- Self-check: Section 3
check("the mean test blocks the worse version",
      lambda: false_pass_risk()["blocked_by_mean_test"] is True)
check("a proven-regression test would not have",
      lambda: false_pass_risk()["would_pass_proven_regression_test"] is True,
      "conservative in both directions: it will not confirm an improvement OR a regression")
check("the gate blocks a version that is genuinely better",
      lambda: false_block_risk()["provably_better"] and false_block_risk()["blocked"],
      "that is not a bug in the gate -- it is the gate expressing a budget")
check("and it says exactly what would have to change",
      lambda: any("cost" in r for r in false_block_risk()["reasons"]))
check("the daily cost difference is the real argument",
      lambda: cost_of_shipping("v8 more agents") > 3 * cost_of_shipping(CURRENT))
check("v7 is the only candidate that both ships and does not cost more than the ceiling",
      lambda: [n for n in CANDIDATES if n != CURRENT and gate(n)["ship"]]
              == ["v7 better prompt"])

def _risks():
    print(f"  daily cost at 20,000 cases:")
    for n in CANDIDATES:
        print(f"    {n:20} {cost_of_shipping(n):>10.2f}")
    print()
    fb = false_block_risk()
    print(f"  {fb['version']} is provably better and is blocked.")
    print(f"  Shipping it anyway costs "
          f"{cost_of_shipping(fb['version']) - cost_of_shipping(CURRENT):.2f} more per day.")
    print("  That is now a decision for whoever owns the budget -- which is the point.")
guard(_risks)

## Run it for real

Write the release note the gate implies. This is what a gate is *for*: not to stop people, but to
make the decision explicit and attributable.

In [ ]:
if llm_ready():
    def _release_note():
        rows = "\n".join(
            f"- {n}: mean {summarise(n)['mean']:.1%} (range {summarise(n)['low']:.0%}-"
            f"{summarise(n)['high']:.0%}), cost {summarise(n)['cost']:.4f}/case, "
            f"p95 {summarise(n)['p95']:.1f}s, gate={'SHIP' if gate(n)['ship'] else 'BLOCK'}"
            for n in CANDIDATES)
        reply = ask(
            "Write a short release recommendation for an engineering manager. Say which version "
            "ships, which is blocked and why, and what decision is being escalated.\n\n"
            f"Ceilings agreed in advance: cost {COST_CEILING}/case, p95 {LATENCY_CEILING}s.\n"
            f"Candidates:\n{rows}")
        print(reply.strip()[:700])
    guard(_release_note)

### Read it

The note should say three things: v7 ships, v8 is blocked on cost and latency rather than on
quality, and someone with a budget needs to decide whether v8's quality is worth roughly three
times the daily spend.

Notice what the gate did **not** do: it did not decide that. It made the trade explicit, attached
numbers to both sides, and put it in front of a person &mdash; which is the same shape as the approval
gate in Module 5 and the refusal in Module 6. A good control does not remove the judgement. It
makes sure the judgement is made by someone entitled to make it, before the fact rather than after.

**What you take from Module 7:** one run is a sample; assert on the trajectory as well as the
outcome; keep a span tree, because attribution needs the nesting; diagnose with an ordered ladder;
and put the numbers in a gate rather than on a dashboard.

In [ ]:
score()

## Your turn

1. `QUALITY_TOLERANCE` is 2%. Given the ranges you measured in Section 1, is that tolerance
   meaningful or is it inside the noise? Pick a defensible value and write the sentence you would
   use to justify it.
2. Add a fourth gate: no trajectory assertion from Lab 7.3 may regress. Which candidate does that
   block, and does it change the recommendation?
3. Every gate needs an override path or it gets bypassed. Write it down: who can override, what
   they must record, and what happens on the next release. A gate with no override is a gate
   people route around.